# Mini-RAG mit mehreren PDFs, Seitenquellen und robustem Prompt

Dieses Notebook erweitert das Mini-RAG um drei praktische Upgrades:

1. **Mehrere PDFs gleichzeitig indexieren**
2. **Quellenangaben mit Dateiname und Seitenbezug ausgeben**
3. **Einen systematischen Prompt für verlässlichere Antworten verwenden**



In [ ]:
# Schritt 1: Pakete installieren
# !pip install -U langchain langchain-openai langchain-community langchain-text-splitters pypdf faiss-cpu python-dotenv
print("Pakete sind installiert oder können bei Bedarf über die auskommentierte Zeile installiert werden.")

In [1]:
# Schritt 1b: API-Key aus .env laden
from dotenv import load_dotenv
import os

load_dotenv("/Users/konstantin/Desktop/dateien/openAi/.env")

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY wurde nicht gefunden. Prüfe den Pfad zur .env-Datei.")

print("OPENAI_API_KEY geladen:", api_key[:10] + "...")

OPENAI_API_KEY geladen: sk-proj-V3...


## Schritt 2: Mehrere PDFs gleichzeitig laden

Hier gibst du eine Liste mit PDF-Dateien an. Jede Datei wird eingelesen und mit Metadaten versehen.
Wichtig: Wir speichern zusätzlich den **Dateinamen**, damit wir ihn später in den Quellen angeben können.


In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

# Passe diese Liste an deine lokalen Dateien an
pdf_paths = [
    "/Users/konstantin/Desktop/dateien/openAi/Advanced_Prompting_Guidelines.pdf",
    "/Users/konstantin/Desktop/dateien/openAi/RAG_Konzept_erklaert.pdf",
    "/Users/konstantin/Desktop/dateien/openAi/Azure_Cloud_Basics_fuer_Interviews.pdf",
]

all_docs = []

for pdf_path in pdf_paths:
    path_obj = Path(pdf_path)
    if not path_obj.exists():
        print(f"Datei nicht gefunden: {pdf_path}")
        continue

    loader = PyPDFLoader(str(path_obj))
    docs = loader.load()

    # Dateiname sauber in die Metadaten schreiben
    for d in docs:
        d.metadata["source_file"] = path_obj.name

    all_docs.extend(docs)
    print(f"Geladen: {path_obj.name} | Seiten: {len(docs)}")

print(f"\nGesamtzahl geladener Seiten: {len(all_docs)}")

# Vorschau auf die erste Seite
if all_docs:
    print("\nErste Quelle:", all_docs[0].metadata)
    print("\nTextvorschau:\n", all_docs[0].page_content[:500])

Geladen: Advanced_Prompting_Guidelines.pdf | Seiten: 2
Geladen: RAG_Konzept_erklaert.pdf | Seiten: 2
Geladen: Azure_Cloud_Basics_fuer_Interviews.pdf | Seiten: 2

Gesamtzahl geladener Seiten: 6

Erste Quelle: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-15T10:14:14+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-04-15T10:14:14+00:00', 'subject': '(unspecified)', 'title': 'Advanced Prompting', 'trapped': '/False', 'source': '/Users/konstantin/Desktop/dateien/openAi/Advanced_Prompting_Guidelines.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Advanced_Prompting_Guidelines.pdf'}

Textvorschau:
 Erstellt für die Interview-Vorbereitung
Seite 1
 Advanced Prompting
Praktische Leitlinien für bessere, verlässlichere Prompts
1. Ziel von gutem Prompting
Ein guter Prompt reduziert Unklarheit. Er sagt dem Modell, welche Aufgabe es hat, welchen
Kontext es nutzen soll, welches Ergebnis erwartet wird und wel

## Schritt 3: Text in Chunks zerlegen

Wir zerlegen die Dokumente in kleinere Textabschnitte.  
Die Metadaten wie Dateiname und Seitenzahl bleiben dabei erhalten.


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = splitter.split_documents(all_docs)

print(f"Anzahl Chunks: {len(chunks)}")
print("\nMetadaten des ersten Chunks:")
print(chunks[0].metadata)

print("\nTextvorschau:")
print(chunks[0].page_content[:400])

Anzahl Chunks: 14

Metadaten des ersten Chunks:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-15T10:14:14+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-04-15T10:14:14+00:00', 'subject': '(unspecified)', 'title': 'Advanced Prompting', 'trapped': '/False', 'source': '/Users/konstantin/Desktop/dateien/openAi/Advanced_Prompting_Guidelines.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Advanced_Prompting_Guidelines.pdf'}

Textvorschau:
Erstellt für die Interview-Vorbereitung
Seite 1
 Advanced Prompting
Praktische Leitlinien für bessere, verlässlichere Prompts
1. Ziel von gutem Prompting
Ein guter Prompt reduziert Unklarheit. Er sagt dem Modell, welche Aufgabe es hat, welchen
Kontext es nutzen soll, welches Ergebnis erwartet wird und welche Grenzen gelten. Advanced
Prompting ist deshalb weniger Magie als saubere Arbeitsanweisung.


## Schritt 4: Embeddings und Vektorindex bauen

Wir verwenden OpenAI-Embeddings und speichern die Chunks in einem FAISS-Index.


In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)

print("FAISS-Index erfolgreich erstellt.")

FAISS-Index erfolgreich erstellt.


## Schritt 5: Retriever bauen

Mit `k=5` holen wir die fünf relevantesten Chunks.  
Das ist oft ein guter Startwert für kleine bis mittlere Dokumentmengen.


In [5]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("Retriever ist bereit.")

Retriever ist bereit.


## Schritt 6: Frage stellen und relevante Treffer anzeigen

Hier prüfen wir zuerst, welche Textstellen das System als relevant findet.


In [6]:
question = "Welche Themen werden in den Dokumenten beschrieben?"

relevant_docs = retriever.invoke(question)

for i, doc in enumerate(relevant_docs, start=1):
    source_file = doc.metadata.get("source_file", "Unbekannte Datei")
    # PyPDFLoader nutzt meist 0-basierte Seitenzahlen in metadata['page']
    page_num = doc.metadata.get("page", None)
    page_display = page_num + 1 if isinstance(page_num, int) else "?"

    print(f"\n--- Treffer {i} | Quelle: {source_file}, Seite {page_display} ---")
    print(doc.page_content[:700])


--- Treffer 1 | Quelle: RAG_Konzept_erklaert.pdf, Seite 1 ---
Vektordatenbank gesucht.

Kontext: Die gefundenen Abschnitte werden als Zusatzinformation an das LLM übergeben.

LLM: Das Modell formuliert daraus eine Antwort, idealerweise mit Bezug auf die gefundenen
Quellen.
3. Warum man Texte in Abschnitte zerlegt
Dokumente werden fast nie als ein einziger großer Block verarbeitet. Stattdessen werden sie in
kleinere Chunks zerlegt. Das verbessert die Trefferqualität, senkt Kosten und hilft dem Modell, nur
den relevanten Teil eines Dokuments zu sehen.

Zu große Chunks: viel irrelevanter Ballast im Kontext

Zu kleine Chunks: wichtige Informationen werden auseinandergerissen

Praxisregel: mittlere Chunk-Größe plus optional leichte Überlappung
4. Was E

--- Treffer 2 | Quelle: RAG_Konzept_erklaert.pdf, Seite 2 ---
9. Gute Interview-Formulierung
„RAG verbindet ein Sprachmodell mit einer semantischen Suche über eigene Dokumente. Dazu
werden Texte in Chunks zerlegt, in Embeddings umgewan

## Schritt 7: Hilfsfunktion für saubere Quellenangaben

Diese Funktion sammelt eindeutige Quellen und formatiert sie lesbar.


In [7]:
def format_sources(docs):
    seen = set()
    sources = []

    for doc in docs:
        source_file = doc.metadata.get("source_file", "Unbekannte Datei")
        page_num = doc.metadata.get("page", None)
        page_display = page_num + 1 if isinstance(page_num, int) else "?"
        key = (source_file, page_display)

        if key not in seen:
            seen.add(key)
            sources.append(f"- {source_file}, Seite {page_display}")

    return "\n".join(sources)

## Schritt 8: Systematischen Prompt für robustere Antworten bauen

Dieser Prompt macht drei Dinge:
- Er zwingt das Modell, **nur auf Basis des Kontexts** zu antworten
- Er verlangt **Unsicherheitsmarkierung**, wenn etwas nicht belegt ist
- Er gibt ein **klares Ausgabeformat** vor


In [8]:
from openai import OpenAI

client = OpenAI()

context = "\n\n".join([
    f"[Quelle: {doc.metadata.get('source_file', 'Unbekannt')}, Seite {(doc.metadata.get('page', '?') + 1) if isinstance(doc.metadata.get('page', None), int) else '?'}]\n{doc.page_content}"
    for doc in relevant_docs
])

sources_text = format_sources(relevant_docs)

prompt = f"""
Du bist ein präziser fachlicher Analyst.

Arbeitsregeln:
1. Beantworte die Frage ausschließlich auf Basis des bereitgestellten Kontexts.
2. Wenn eine Information nicht im Kontext enthalten ist, sage das ausdrücklich.
3. Erfinde keine Fakten und keine Quellen.
4. Formuliere klar, knapp und nachvollziehbar.
5. Nenne am Ende die verwendeten Quellen mit Dateiname und Seitenzahl.

Ausgabeformat:
- Kurze Antwort
- Zentrale Punkte als Bulletpoints
- Abschnitt 'Quellen'

Frage:
{question}

Kontext:
{context}

Verfügbare Quellen:
{sources_text}
"""

response = client.responses.create(
    model="gpt-4.1-mini",
    input=prompt
)

print(response.output_text)

- Kurze Antwort  
Die Dokumente beschreiben Konzepte und Methoden zur Nutzung von Sprachmodellen, insbesondere den Einsatz von Vektordatenbanken, Textzerlegung in Chunks, Embeddings und semantischer Suche (RAG-Konzept). Zudem enthalten sie Richtlinien für professionelle Promptgestaltung und Interview-Vorbereitung im Bereich Azure Cloud und Advanced Prompting.

- Zentrale Punkte  
  - RAG-Konzept (Retrieval-Augmented Generation):  
    - Zerlegung von Texten in passende Abschnitte (Chunks) zur besseren Verarbeitung.  
    - Nutzung von Embeddings zur semantischen Vektorensuche.  
    - Kombination von Sprachmodell und Dokumentensuche für fundierte Antworten.  
  - Advanced Prompting Guidelines:  
    - Vorgaben für Ausgabeformate (z.B. Tabellen, Bulletpoints).  
    - Qualitätskriterien wie präzise, quellenbasierte Antworten.  
    - Formen der Sicherheit und Transparenz bei Antworten (z.B. Unsicherheitsmarkierung).  
    - Beispiele für schwache und starke Prompts zur besseren Steuerun

## Optional: Eine wiederverwendbare Frage-Funktion

Mit dieser Funktion kannst du später beliebige Fragen an dein Mini-RAG stellen.


In [9]:
def ask_rag(question, retriever, client, model="gpt-4.1-mini", k=5):
    # temporär eigenen Retriever mit k erzeugen
    local_retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    relevant_docs = local_retriever.invoke(question)

    context = "\n\n".join([
        f"[Quelle: {doc.metadata.get('source_file', 'Unbekannt')}, Seite {(doc.metadata.get('page', '?') + 1) if isinstance(doc.metadata.get('page', None), int) else '?'}]\n{doc.page_content}"
        for doc in relevant_docs
    ])

    sources_text = format_sources(relevant_docs)

    prompt = f"""
Du bist ein präziser fachlicher Analyst.

Arbeitsregeln:
1. Beantworte die Frage ausschließlich auf Basis des bereitgestellten Kontexts.
2. Wenn eine Information nicht im Kontext enthalten ist, sage das ausdrücklich.
3. Erfinde keine Fakten und keine Quellen.
4. Formuliere klar, knapp und nachvollziehbar.
5. Nenne am Ende die verwendeten Quellen mit Dateiname und Seitenzahl.

Ausgabeformat:
- Kurze Antwort
- Zentrale Punkte als Bulletpoints
- Abschnitt 'Quellen'

Frage:
{question}

Kontext:
{context}

Verfügbare Quellen:
{sources_text}
"""

    response = client.responses.create(
        model=model,
        input=prompt
    )

    return response.output_text, relevant_docs

In [10]:
# Beispielaufruf
answer, retrieved = ask_rag(
    question="Welche Unterschiede zwischen RAG, Azure/Cloud Basics und Advanced Prompting werden in den Dokumenten thematisiert?",
    retriever=retriever,
    client=client,
    model="gpt-4.1-mini",
    k=5
)

print(answer)

- **RAG (Retrieval-Augmented Generation)**  
  - Verbindet Sprachmodell mit semantischer Suche über eigene Dokumente  
  - Ermöglicht Antworten mit Zugriff auf aktuelle, interne Inhalte  
  - Typische Bausteine: Dokumenten-Loader, Text-Splitting, Embedding-Modell, Vektordatenbank, Retriever, LLM  
  - Schwächen: schlechte Chunking-Strategie, irrelevante Quellen, fehlende Quellenangaben, keine Qualitätskontrolle  
  - Fokus liegt auf Quellenbezug und Transparenz der Antworten  

- **Azure/Cloud Basics**  
  - Überblick über Cloud-Konzepte: zentrale Bereitstellung von Rechenleistung, Speicher und Diensten  
  - Beispiele für Azure-Dienste: Azure OpenAI, Azure Machine Learning, Databricks, M365/Copilot  
  - Fokus auf Verständnis der Architektur: Wo Daten und Modelle laufen, Sicherheit, Betrieb und Monitoring  
  - Ziel: Wichtige Cloud- und Azure-Begriffe und -Dienste für Interviews erklären können  

- **Advanced Prompting**  
  - Leitlinien für präzise, klare und zuverlässige Prompts an